In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import kagglehub
import os
import unicodedata
import hashlib
import sacrebleu as sbleu

from kagglehub import KaggleDatasetAdapter
from itertools import chain
from regex import regex as re
from langdetect import detect, DetectorFactory
from rouge_score import rouge_scorer
from collections import defaultdict
from typing import Dict
from datasets import load_dataset, DownloadMode
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

from scripts.libs.utils import normalize_text,  sample_lang_distribution
from scripts.libs.tldr_stats import compute_tldr_stats, compute_alignment_stats, tokenize_texts, evaluate_echo_baseline
from scripts.libs.llm_client import LLMClient
from scripts.libs.slang_annotator import SlangAnnotator
from scripts.libs.analyse_slang_dataset import SlangAnalyzer
from scripts.libs.top_p import get_top_p_indices

tqdm.pandas()

client = LLMClient()
analyser = SlangAnalyzer()
basepath = "Data/"
gen_zz_words = "gen_zz_words.csv"
genz_slang_pairs = "genz-slang-pairs-1k.csv"
genz_slang = "genz_slang.csv"

In [10]:
print("Loading TL;DR dataset from Hugging Face...")
dataset = load_dataset("trl-lib/tldr", download_mode=DownloadMode.REUSE_CACHE_IF_EXISTS)

train_ds = dataset["train"]
val_ds = dataset["validation"]
test_ds = dataset["test"]

print(f"Dataset sizes: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")
total = len(train_ds) + len(val_ds) + len(test_ds)
print(f"Dataset split percentages: Train={len(train_ds)/total*100:.2f}%, Val={len(val_ds)/total*100:.2f}%, Test={len(test_ds)/total*100:.2f}%")

pd.DataFrame(train_ds).head(5)

Loading TL;DR dataset from Hugging Face...


Generating test split: 100%|██████████| 6553/6553 [00:00<00:00, 139391.79 examples/s]


Dataset sizes: Train=116722, Val=6447, Test=6553
Dataset split percentages: Train=89.98%, Val=4.97%, Test=5.05%


,prompt,completion
0,SUBREDDIT: r/relationships\n\nTITLE: I (f/22) ...,I still have contact with an old ex's friends...
1,SUBREDDIT: r/loseit\n\nTITLE: SV & NSV! Keepin...,"Progress is still happening, even when you th..."
2,SUBREDDIT: r/relationships\n\nTITLE: Me [19F] ...,My skin is scarred badly; what could I do/say...
3,SUBREDDIT: r/personalfinance\n\nTITLE: Priorit...,$14k in student debt (all <5%) and need to sa...
4,SUBREDDIT: r/relationships\n\nTITLE: My[25m] g...,"GF is a meanie-bo-beanie when I'm nice, and a..."


In [11]:
train_ds = train_ds.map(normalize_text, desc="Normalizing train")
val_ds = val_ds.map(normalize_text, desc="Normalizing validation")
test_ds = test_ds.map(normalize_text, desc="Normalizing test")

Normalizing test: 100%|██████████| 6553/6553 [00:01<00:00, 6072.99 examples/s]


In [12]:
df_genz = pd.read_csv(basepath + "genz_slang_data.csv")
df_genz.head(5)

,Slang,Description,Example,Context,Neutral_Text
0,"""hella""",Intensifier meaning extremely or very,I'm hella drained today.,"Typically expressed in casual conversations, p...",I'm feeling really tired today.
1,"""mad""","Angry or frustrated, often informally.",I'm mad tired today 'cause I pulled an all-nig...,Typically expressed among peers or casual acqu...,I'm really tired today because I stayed up lat...
2,Chillin',"Casually relaxing, informally enjoying leisure","Fr, I’m just chillin’ at home tonight, no cap.",Typically utilized among friends or peers duri...,"Yeah, I think I'm going to stay in tonight and..."
3,vibing,"Enjoying oneself in a relaxed, positive manner...","Yo, I'm just chillin' at my place and vibing t...","In casual social settings, often among friends...","Hey, I'm just relaxing at home and listening t..."
4,OOTD,Outfit of the day,"OOTD, just vibing with the crew today, no cap.",Typically exchanged among young adults on soci...,I'm just planning to hang out with friends thi...


In [13]:
# Load model and embeddings (do this once)
model = SentenceTransformer("all-MiniLM-L6-v2")
slang_embeddings = model.encode(df_genz['Neutral_Text'].tolist(), show_progress_bar=True)

Batches: 100%|██████████| 60/60 [00:31<00:00,  1.89it/s]


In [14]:
model = SentenceTransformer("all-MiniLM-L6-v2")
slang_embeddings = model.encode(df_genz['Neutral_Text'].tolist(), show_progress_bar=True)

Batches: 100%|██████████| 60/60 [00:29<00:00,  2.00it/s]


In [17]:
def analyze_slang_similarity(tldr_text, top_p):
    """
    For one TL;DR post:
    - Computes similarity against all slangs
    - Returns a DataFrame with: Slang, Similarity score, IsRelevant
    """
    tldr_embedding = model.encode([tldr_text])[0]
    similarities = cosine_similarity([tldr_embedding], slang_embeddings)[0]
    similarities = (similarities - similarities.min()) / (similarities.max() - similarities.min())

    # Top-p selection
    top_p_indices = get_top_p_indices(similarities, top_p=top_p)
    top_p_mask = np.zeros_like(similarities, dtype=bool)
    top_p_mask[top_p_indices] = True

    # Build result DataFrame
    df_result = pd.DataFrame({
        'Slang': df_genz['Slang'].tolist(),
        'Similarity': similarities,
        'Relevant': top_p_mask
    })
    return df_result

In [37]:
tldr_df_train = train_ds.to_pandas().head(1).copy()
slang_analysis_results = []

for idx, completion in tqdm(
        enumerate(tldr_df_train['completion']),
        total=len(tldr_df_train),
        desc="Analyzing slang similarities"
):
    df_slang = analyze_slang_similarity(completion, top_p=0.01)
    slang_analysis_results.append(df_slang)

similarities = (
    df_slang.loc[df_slang['Relevant'] == True]
    .sort_values('Similarity')
    .reset_index(drop=True)
)

display(similarities)


Analyzing slang similarities: 100%|██████████| 1/1 [00:00<00:00,  3.19it/s]


,Slang,Similarity,Relevant
0,Breadcumbing,0.733911,True
1,BFFLNMW,0.748868,True
2,FB,0.763188,True
3,MU,0.787205,True
4,BFFL,0.809796,True
5,FBO,0.814704,True
6,FWB,0.830394,True
7,AAYF,0.833631,True
8,Thirsty,0.854736,True
9,BFFN,0.861697,True


In [ ]:
# If you want, add summary stats per post
# tldr_df_train['relevant_slang_count'] = [df['Relevant'].sum() for df in slang_analysis_results]

# print(f"Relevant slang count - Mean: {tldr_df_train['relevant_slang_count'].mean():.2f}, "
#       f"Min: {tldr_df_train['relevant_slang_count'].min()}, "
#       f"Max: {tldr_df_train['relevant_slang_count'].max()}")
# print(df_slang['Similarity'].max())
# print(df_slang['Similarity'].min())
# display(df_slang['Similarity'].sort_values())

# Save annotated main data
# tldr_df_train.to_csv("tldr_with_slang.csv", index=False)
# print(f"\nSaved {len(tldr_df_train)} Reddit posts with all slang analyses.")